# Accuracy-Focused Runtime Notebook

This notebook is built for one goal: improve runtime-prediction accuracy on the slow tail.

It supports both:
- `MODEL_VARIANT = "normal"`
- `MODEL_VARIANT = "cross_attention"`

Key choices in this notebook:
- weighted sampling / oversampling for slow programs
- `MAX_LENGTH = 128`
- unfreeze only the last CodeBERT layer
- lower backbone LR and higher head LR

In [ ]:
import json
import math
import os
import time

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.optim import AdamW
from torch.utils.data import DataLoader, Dataset, WeightedRandomSampler
from transformers import AutoModel, AutoTokenizer

MODEL_VARIANT = 'cross_attention'   # 'normal' or 'cross_attention'
BATCH_SIZE = 32
EPOCHS = 20
HEAD_LEARNING_RATE = 1e-4
BACKBONE_LEARNING_RATE = 5e-6
MAX_LENGTH = 128
RANDOM_SEED = 42
UNFROZEN_CODEBERT_LAYERS = 1
DATA_FILE = 'Cleaned_LOOPerSet_GEM.csv'
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
FEATURE_COLS = [
    'T1', 'T2', 'Unroll', 'locality', 'working_set_size',
    'arithmetic_intensity', 'T1_T2_inter', 'log_T1', 'log_T2', 'T1_div_T2'
]

NORMAL_MODEL_PATH = 'accuracy_normal_model.pth'
CROSS_MODEL_PATH = 'accuracy_cross_attention_model.pth'
NORMAL_METRICS_PATH = 'accuracy_normal_metrics.json'
CROSS_METRICS_PATH = 'accuracy_cross_attention_metrics.json'

torch.manual_seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)
print(f'[INFO] Device: {DEVICE}')
print(f'[INFO] Model variant: {MODEL_VARIANT}')

In [ ]:
def compute_regression_metrics(actual_ms, predicted_ms):
    actual_ms = np.asarray(actual_ms, dtype=np.float64)
    predicted_ms = np.asarray(predicted_ms, dtype=np.float64)
    residuals = predicted_ms - actual_ms
    absolute_errors = np.abs(residuals)
    mae = float(np.mean(absolute_errors))
    rmse = float(np.sqrt(np.mean(np.square(residuals))))
    median_ae = float(np.median(absolute_errors))
    safe_actual = np.clip(np.abs(actual_ms), 1e-6, None)
    mape = float(np.mean(absolute_errors / safe_actual) * 100.0)
    ss_res = float(np.sum(np.square(residuals)))
    ss_tot = float(np.sum(np.square(actual_ms - actual_ms.mean())))
    r2 = float(1.0 - (ss_res / ss_tot)) if ss_tot > 0 else float('nan')
    return {
        'mae_ms': mae,
        'rmse_ms': rmse,
        'median_ae_ms': median_ae,
        'mape_percent': mape,
        'r2': r2,
        'residuals': residuals,
        'absolute_errors': absolute_errors,
    }


def evaluate_model(model, data_loader, criterion):
    model.eval()
    total_loss = 0.0
    predictions_log = []
    labels_log = []
    with torch.no_grad():
        for batch in data_loader:
            input_ids = batch['input_ids'].to(DEVICE)
            attention_mask = batch['attention_mask'].to(DEVICE)
            schedule_features = batch['schedule_features'].to(DEVICE)
            labels = batch['labels'].to(DEVICE)
            outputs = model(input_ids, attention_mask, schedule_features).squeeze(-1)
            loss = criterion(outputs, labels)
            total_loss += loss.item()
            predictions_log.append(outputs.cpu().numpy())
            labels_log.append(labels.cpu().numpy())

    predictions_log = np.concatenate(predictions_log)
    labels_log = np.concatenate(labels_log)
    predicted_ms = np.expm1(predictions_log)
    actual_ms = np.expm1(labels_log)
    metrics = compute_regression_metrics(actual_ms, predicted_ms)
    metrics['avg_loss'] = total_loss / max(len(data_loader), 1)
    metrics['predicted_ms'] = predicted_ms
    metrics['actual_ms'] = actual_ms
    return metrics

In [ ]:
class CompilerDataset(Dataset):
    def __init__(self, dataframe, tokenizer, feature_means, feature_stds, max_length=128):
        self.df = dataframe.reset_index(drop=True)
        self.tokenizer = tokenizer
        self.max_length = max_length
        self.feature_means = np.asarray(feature_means, dtype=np.float32)
        self.feature_stds = np.asarray(feature_stds, dtype=np.float32)

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        encoding = self.tokenizer(
            row['Code_String'],
            return_tensors='pt',
            max_length=self.max_length,
            padding='max_length',
            truncation=True,
        )
        raw_features = np.array([
            float(row['T1']),
            float(row['T2']),
            float(row['Unroll']),
            float(row['locality']),
            float(row['working_set_size']),
            float(row['arithmetic_intensity']),
            float(row['T1_T2_inter']),
            float(row['log_T1']),
            float(row['log_T2']),
            float(row['T1_div_T2']),
        ], dtype=np.float32)
        scaled_features = (raw_features - self.feature_means) / self.feature_stds
        label = float(row['log_execution_time'])
        return {
            'input_ids': encoding['input_ids'].squeeze(0),
            'attention_mask': encoding['attention_mask'].squeeze(0),
            'schedule_features': torch.tensor(scaled_features, dtype=torch.float32),
            'labels': torch.tensor(label, dtype=torch.float32),
        }

In [ ]:
class NormalCostModel(nn.Module):
    def __init__(self, schedule_feature_size=10):
        super().__init__()
        self.codebert = AutoModel.from_pretrained('microsoft/codebert-base')
        self.schedule_mlp = nn.Sequential(
            nn.Linear(schedule_feature_size, 64),
            nn.GELU(),
            nn.Dropout(0.1),
            nn.Linear(64, 32),
            nn.GELU(),
        )
        self.regressor = nn.Sequential(
            nn.Linear(768 + 32, 128),
            nn.GELU(),
            nn.Dropout(0.1),
            nn.Linear(128, 1),
        )

    def forward(self, input_ids, attention_mask, schedule_features):
        code_outputs = self.codebert(input_ids=input_ids, attention_mask=attention_mask)
        token_embeddings = code_outputs.last_hidden_state
        mask = attention_mask.unsqueeze(-1).float()
        pooled_code = (token_embeddings * mask).sum(dim=1) / mask.sum(dim=1).clamp(min=1.0)
        schedule_embedding = self.schedule_mlp(schedule_features)
        fused = torch.cat((pooled_code, schedule_embedding), dim=1)
        return self.regressor(fused)


class CrossAttentionCostModel(nn.Module):
    def __init__(self, schedule_feature_size=10, attention_dim=128):
        super().__init__()
        self.codebert = AutoModel.from_pretrained('microsoft/codebert-base')
        self.schedule_mlp = nn.Sequential(
            nn.Linear(schedule_feature_size, 64),
            nn.GELU(),
            nn.Dropout(0.1),
            nn.Linear(64, 32),
            nn.GELU(),
        )
        self.query_proj = nn.Linear(32, attention_dim)
        self.key_proj = nn.Linear(768, attention_dim)
        self.value_proj = nn.Linear(768, attention_dim)
        self.regressor = nn.Sequential(
            nn.Linear(attention_dim + 32, 128),
            nn.GELU(),
            nn.Dropout(0.1),
            nn.Linear(128, 1),
        )

    def forward(self, input_ids, attention_mask, schedule_features, return_attention=False):
        code_outputs = self.codebert(input_ids=input_ids, attention_mask=attention_mask)
        token_embeddings = code_outputs.last_hidden_state
        schedule_embedding = self.schedule_mlp(schedule_features)
        query = self.query_proj(schedule_embedding).unsqueeze(1)
        keys = self.key_proj(token_embeddings)
        values = self.value_proj(token_embeddings)
        scores = torch.matmul(query, keys.transpose(1, 2)) / math.sqrt(keys.size(-1))
        mask = attention_mask.unsqueeze(1) == 0
        scores = scores.masked_fill(mask, float('-inf'))
        attention_weights = torch.softmax(scores, dim=-1)
        attended_code = torch.matmul(attention_weights, values).squeeze(1)
        fused = torch.cat((attended_code, schedule_embedding), dim=1)
        prediction = self.regressor(fused)
        if return_attention:
            return prediction, attention_weights.squeeze(1)
        return prediction

In [ ]:
print('[INFO] Loading tokenizer and dataset...')
tokenizer = AutoTokenizer.from_pretrained('microsoft/codebert-base')
full_df = pd.read_csv(DATA_FILE)
num_rows = len(full_df)

generator = torch.Generator().manual_seed(RANDOM_SEED)
permutation = torch.randperm(num_rows, generator=generator).tolist()
train_size = int(0.8 * num_rows)
remaining_size = num_rows - train_size
val_size = remaining_size // 2

a_idx = permutation[:train_size]
b_idx = permutation[train_size:train_size + val_size]
c_idx = permutation[train_size + val_size:]

train_df = full_df.iloc[a_idx].reset_index(drop=True)
val_df = full_df.iloc[b_idx].reset_index(drop=True)
test_df = full_df.iloc[c_idx].reset_index(drop=True)

feature_means = train_df[FEATURE_COLS].mean().to_numpy(dtype=np.float32)
feature_stds = train_df[FEATURE_COLS].std().replace(0, 1.0).to_numpy(dtype=np.float32)
feature_stats = {'means': feature_means.tolist(), 'stds': feature_stds.tolist(), 'feature_cols': FEATURE_COLS}

# Weighted sampling to oversample slow programs.
# We emphasize the regions that hurt accuracy most in earlier runs.
train_exec = train_df['execution_time'].to_numpy()
train_weights = np.ones(len(train_df), dtype=np.float32)
train_weights[(train_exec > 1000) & (train_exec <= 10000)] = 3.0   # 1s to 10s
train_weights[train_exec > 10000] = 6.0                             # >10s
weighted_sampler = WeightedRandomSampler(
    weights=torch.as_tensor(train_weights, dtype=torch.double),
    num_samples=len(train_weights),
    replacement=True,
)

train_dataset = CompilerDataset(train_df, tokenizer, feature_means, feature_stds, max_length=MAX_LENGTH)
val_dataset = CompilerDataset(val_df, tokenizer, feature_means, feature_stds, max_length=MAX_LENGTH)
test_dataset = CompilerDataset(test_df, tokenizer, feature_means, feature_stds, max_length=MAX_LENGTH)

loader_kwargs = {'batch_size': BATCH_SIZE, 'num_workers': 2, 'pin_memory': torch.cuda.is_available()}
train_loader = DataLoader(train_dataset, sampler=weighted_sampler, shuffle=False, **loader_kwargs)
val_loader = DataLoader(val_dataset, shuffle=False, **loader_kwargs)
test_loader = DataLoader(test_dataset, shuffle=False, **loader_kwargs)

slow_1_to_10s = int(((train_exec > 1000) & (train_exec <= 10000)).sum())
slow_gt_10s = int((train_exec > 10000).sum())
print(f'[INFO] Split sizes -> train: {len(train_df)}, val: {len(val_df)}, test: {len(test_df)}')
print(f'[INFO] Weighted sampler slow buckets -> 1s-10s: {slow_1_to_10s}, >10s: {slow_gt_10s}')
print('[INFO] Example feature means:', feature_means[:3])
print('[INFO] Example feature stds:', feature_stds[:3])

In [ ]:
if MODEL_VARIANT == 'normal':
    model = NormalCostModel(schedule_feature_size=len(FEATURE_COLS)).to(DEVICE)
    BEST_MODEL_PATH = NORMAL_MODEL_PATH
    METRICS_PATH = NORMAL_METRICS_PATH
elif MODEL_VARIANT == 'cross_attention':
    model = CrossAttentionCostModel(schedule_feature_size=len(FEATURE_COLS)).to(DEVICE)
    BEST_MODEL_PATH = CROSS_MODEL_PATH
    METRICS_PATH = CROSS_METRICS_PATH
else:
    raise ValueError("MODEL_VARIANT must be 'normal' or 'cross_attention'")

for param in model.codebert.parameters():
    param.requires_grad = False

encoder_layers = model.codebert.encoder.layer
for layer in encoder_layers[-UNFROZEN_CODEBERT_LAYERS:]:
    for param in layer.parameters():
        param.requires_grad = True

backbone_params = []
head_params = []
for name, param in model.named_parameters():
    if not param.requires_grad:
        continue
    if name.startswith('codebert.'):
        backbone_params.append(param)
    else:
        head_params.append(param)

optimizer = AdamW([
    {'params': backbone_params, 'lr': BACKBONE_LEARNING_RATE},
    {'params': head_params, 'lr': HEAD_LEARNING_RATE},
])
criterion = nn.MSELoss()

trainable_total = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'[INFO] Model ready: {MODEL_VARIANT}')
print(f'[INFO] Trainable params: {trainable_total:,}')
print(f'[INFO] Backbone LR: {BACKBONE_LEARNING_RATE} | Head LR: {HEAD_LEARNING_RATE}')

In [ ]:
train_losses = []
val_losses = []
val_maes = []
val_r2s = []
backbone_lr_history = []
head_lr_history = []
best_val_loss = float('inf')

for epoch in range(EPOCHS):
    model.train()
    running_loss = 0.0

    for batch in train_loader:
        input_ids = batch['input_ids'].to(DEVICE)
        attention_mask = batch['attention_mask'].to(DEVICE)
        schedule_features = batch['schedule_features'].to(DEVICE)
        labels = batch['labels'].to(DEVICE)

        optimizer.zero_grad()
        outputs = model(input_ids, attention_mask, schedule_features).squeeze(-1)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        running_loss += loss.item()

    avg_train_loss = running_loss / max(len(train_loader), 1)
    val_metrics = evaluate_model(model, val_loader, criterion)

    train_losses.append(avg_train_loss)
    val_losses.append(val_metrics['avg_loss'])
    val_maes.append(val_metrics['mae_ms'])
    val_r2s.append(val_metrics['r2'])
    backbone_lr_history.append(optimizer.param_groups[0]['lr'])
    head_lr_history.append(optimizer.param_groups[1]['lr'])

    print(
        f"Epoch {epoch + 1:02d}/{EPOCHS} | "
        f"train_loss={avg_train_loss:.4f} | val_loss={val_metrics['avg_loss']:.4f} | "
        f"val_mae={val_metrics['mae_ms']:.2f} ms | val_r2={val_metrics['r2']:.4f}"
    )

    if val_metrics['avg_loss'] < best_val_loss:
        best_val_loss = val_metrics['avg_loss']
        torch.save({'model_state_dict': model.state_dict(), 'feature_stats': feature_stats}, BEST_MODEL_PATH)
        print('  [INFO] Saved new best checkpoint.')

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
epochs = range(1, EPOCHS + 1)

axes[0, 0].plot(epochs, train_losses, marker='o', label='Train Loss')
axes[0, 0].plot(epochs, val_losses, marker='s', label='Val Loss')
axes[0, 0].set_title('Train vs Validation Loss')
axes[0, 0].legend()
axes[0, 0].grid(True, linestyle='--', alpha=0.4)

axes[0, 1].plot(epochs, val_maes, marker='o', color='darkorange')
axes[0, 1].set_title('Validation MAE')
axes[0, 1].grid(True, linestyle='--', alpha=0.4)

axes[1, 0].plot(epochs, val_r2s, marker='o', color='seagreen')
axes[1, 0].set_title('Validation Accuracy (R^2)')
axes[1, 0].grid(True, linestyle='--', alpha=0.4)

axes[1, 1].plot(epochs, backbone_lr_history, marker='o', label='Backbone LR')
axes[1, 1].plot(epochs, head_lr_history, marker='s', label='Head LR')
axes[1, 1].set_title('Learning Rates')
axes[1, 1].legend()
axes[1, 1].grid(True, linestyle='--', alpha=0.4)

plt.tight_layout()
plt.show()

In [ ]:
checkpoint = torch.load(BEST_MODEL_PATH, map_location=DEVICE)
model.load_state_dict(checkpoint['model_state_dict'])
model.eval()
print('[INFO] Best checkpoint reloaded for test evaluation.')

test_metrics = evaluate_model(model, test_loader, criterion)
metrics_to_save = {
    'model_variant': MODEL_VARIANT,
    'mae_ms': test_metrics['mae_ms'],
    'rmse_ms': test_metrics['rmse_ms'],
    'median_ae_ms': test_metrics['median_ae_ms'],
    'mape_percent': test_metrics['mape_percent'],
    'r2': test_metrics['r2'],
}
with open(METRICS_PATH, 'w', encoding='utf-8') as f:
    json.dump(metrics_to_save, f, indent=2)

print('Test metrics:')
for key, value in metrics_to_save.items():
    print(f'  {key}: {value}')

In [ ]:
actual_ms = test_metrics['actual_ms']
predicted_ms = test_metrics['predicted_ms']
residuals = test_metrics['residuals']

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
axes[0].scatter(actual_ms, predicted_ms, alpha=0.35)
min_val = max(min(actual_ms.min(), predicted_ms.min()), 1e-6)
max_val = max(actual_ms.max(), predicted_ms.max())
axes[0].plot([min_val, max_val], [min_val, max_val], color='red', linestyle='--')
axes[0].set_xscale('log')
axes[0].set_yscale('log')
axes[0].set_title('Predicted vs Actual')

axes[1].scatter(actual_ms, residuals, alpha=0.35, color='darkorange')
axes[1].axhline(0.0, color='black', linestyle='--')
axes[1].set_xscale('log')
axes[1].set_title('Residuals vs Actual')

axes[2].hist(residuals, bins=60, density=True, alpha=0.75, color='seagreen')
mu = residuals.mean()
sigma = residuals.std() if residuals.std() > 0 else 1.0
x = np.linspace(residuals.min(), residuals.max(), 400)
gaussian = (1.0 / (sigma * np.sqrt(2 * np.pi))) * np.exp(-0.5 * ((x - mu) / sigma) ** 2)
axes[2].plot(x, gaussian, color='black', linewidth=2)
axes[2].set_title('Error Distribution')

plt.tight_layout()
plt.show()

In [ ]:
error_df = test_df.copy().reset_index(drop=True)
error_df['actual_ms'] = test_metrics['actual_ms']
error_df['predicted_ms'] = test_metrics['predicted_ms']
error_df['absolute_error_ms'] = np.abs(error_df['predicted_ms'] - error_df['actual_ms'])

runtime_bins = [0, 1, 10, 100, 1000, 10000, np.inf]
runtime_labels = ['<=1 ms', '1-10 ms', '10-100 ms', '100 ms-1 s', '1-10 s', '>10 s']
error_df['runtime_bucket'] = pd.cut(error_df['actual_ms'], bins=runtime_bins, labels=runtime_labels, include_lowest=True)

bucket_summary = error_df.groupby('runtime_bucket', observed=False).agg(
    count=('actual_ms', 'size'),
    mae_ms=('absolute_error_ms', 'mean'),
    median_ae_ms=('absolute_error_ms', 'median'),
).reset_index()

print('Worst 10 predictions by absolute error:')
display(error_df[['Code_String', 'actual_ms', 'predicted_ms', 'absolute_error_ms', 'T1', 'T2', 'Unroll']].sort_values('absolute_error_ms', ascending=False).head(10))
print('Runtime bucket summary:')
display(bucket_summary)